In [10]:
import time
import re
import random
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from concurrent.futures import ThreadPoolExecutor

BASE_URL = "https://www.polsinelli.com"
VALID_BAR_ADMISSIONS = {
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware",
    "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky",
    "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico",
    "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
    "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
    "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming", "District of Columbia",
    "USPTO"
}

OUTPUT_COLUMNS = [
    'Company Name', 'First Name', 'Middle Initial', 'Last Name', 'Job Title', 'Company Email',
    'Attorney Location(s)', 'Work Phone Number', 'Biography/Overview', 'Practice Group', 'Specialties',
    'Industry Focus', 'Law School Name', 'Law School Graduation Year', 'Law School Honors',
    'Undergraduate School Name', 'Undergraduate Graduation Year', 'Undergraduate School Honors',
    'Bar Admissions', 'Languages', 'Attorney Website URL', 'Attorney LinkedIn URL', 'Photo?', 'Date Added to Database'
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36"
]

def split_name(name):
    name = re.sub(r",\s*[A-Za-z\.]+$", "", name).strip()
    parts = name.split()
    if len(parts) == 2:
        return parts[0], "N/A", parts[1]
    elif len(parts) == 3:
        if re.match(r"^([A-Za-z]\.){1,2}$", parts[1]):
            return parts[0], parts[1], parts[2]
        else:
            return parts[0] + " " + parts[1], "N/A", parts[2]
    elif len(parts) > 3:
        return " ".join(parts[:-1]), "N/A", parts[-1]
    else:
        return name, "N/A", "N/A"

def get_profile_urls():
    options = Options()
    options.add_argument("--headless")
    options.add_argument(f"--user-agent={random.choice(USER_AGENTS)}")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    profile_urls = []
    for offset in range(0, 1200, 40):
        url = f"{BASE_URL}/people?search[post_type]=person&from={offset}"
        print(f"🔍 Fetching: {url}")
        driver.get(url)
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        cards = soup.select("a[href^='/'][aria-label^='Visit Biography']")
        for a_tag in cards:
            href = a_tag.get("href")
            if href and href.startswith("/"):
                profile_urls.append(BASE_URL + href)
    driver.quit()
    print(f"🔗 Total profile URLs found: {len(profile_urls)}")
    return profile_urls

def extract_bar_admissions(soup):
    bar_section = soup.find("h3", string=re.compile("Bar Admission", re.I))
    if bar_section:
        ul = bar_section.find_next("ul")
        if ul:
            admissions = [li.get_text(strip=True).split(",")[0] for li in ul.find_all("li")]
            filtered = [a for a in admissions if a in VALID_BAR_ADMISSIONS]
            return ", ".join(filtered) if filtered else "N/A"
    return "N/A"

def extract_specialties(soup):
    h3 = soup.find("h3", string=re.compile("Capabilities", re.I))
    if h3:
        ul = h3.find_next("ul")
        if ul:
            return ", ".join(li.get_text(strip=True) for li in ul.find_all("li"))
    return "N/A"

def extract_education(soup):
    law_name, law_year, law_honors = "N/A", "N/A", "N/A"
    ug_name, ug_year, ug_honors = "N/A", "N/A", "N/A"
    h3 = soup.find("h3", string=re.compile("Education", re.I))
    if h3:
        ul = h3.find_next("ul")
        if ul:
            for li in ul.find_all("li"):
                text = li.get_text(" ", strip=True)
                year_match = re.search(r"(\d{4})", text)
                year = year_match.group(1) if year_match else "N/A"
                if "J.D." in text:
                    law_name = re.sub(r"\s*\(.*", "", text)
                    law_year = year
                    law_honors = "J.D."
                elif re.search(r"B\.[A|S]", text):
                    ug_name = re.sub(r"\s*\(.*", "", text)
                    ug_year = year
                    honors_match = re.search(r"\((.*?)\)", text)
                    ug_honors = honors_match.group(1).split(",")[0] if honors_match else "N/A"
    return law_name, law_year, law_honors, ug_name, ug_year, ug_honors

def scrape_profile(driver, url):
    try:
        print(f"🔎 Scraping: {url}")
        driver.get(url)
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        name_tag = soup.select_one("h1 span")
        full_name = name_tag.get_text(strip=True) if name_tag else "N/A"
        first, middle, last = split_name(full_name)

        title_tag = soup.find("div", class_="sc-gsTCUz hhXTek")
        title = title_tag.get_text(strip=True) if title_tag else "N/A"

        email_tag = soup.select_one("a[href^='mailto']")
        email = email_tag.get_text(strip=True) if email_tag else "N/A"

        phone_tag = soup.select_one("a[href^='tel']")
        phone = phone_tag.get_text(strip=True) if phone_tag else "N/A"

        location_tag = soup.select_one("div.type-office a")
        location = location_tag.get_text(strip=True).replace(":", "") if location_tag else "N/A"

        bio_section = soup.select_one("div.sc-gsTCUz.kRqOzn div.sc-gsTCUz.bPkhFt")
        bio = bio_section.get_text(" ", strip=True) if bio_section else "N/A"

        linkedin_tag = soup.select_one("a[href*='linkedin.com']")
        linkedin = linkedin_tag['href'] if linkedin_tag else "N/A"

        photo_tag = soup.select_one("img[src*='/content/uploads']")
        photo_url = photo_tag["src"] if photo_tag else "N/A"

        bar_admissions = extract_bar_admissions(soup)
        specialties = extract_specialties(soup)
        law_name, law_year, law_honors, ug_name, ug_year, ug_honors = extract_education(soup)
        date_added = datetime.today().strftime("%Y-%m-%d")

        return [
            "Polsinelli", first, middle, last, title, email, location, phone, bio,
            "N/A", specialties, "N/A", law_name, law_year, law_honors,
            ug_name, ug_year, ug_honors, bar_admissions, "N/A",
            url, linkedin, photo_url, date_added
        ]
    except Exception as e:
        print(f"❌ Failed scraping {url}: {e}")
        return None

def scrape_all_profiles():
    urls = get_profile_urls()
    all_data = []

    options = Options()
    options.add_argument(f"--user-agent={random.choice(USER_AGENTS)}")
    options.add_argument("--headless")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        for url in urls:
            row = scrape_profile(driver, url)
            if row:
                all_data.append(row)
    finally:
        driver.quit()

    df = pd.DataFrame(all_data, columns=OUTPUT_COLUMNS)
    df.to_excel("Polsinelli_Attorneys_Full.xlsx", index=False)
    print("✅ Saved to Polsinelli_Attorneys_Full.xlsx")

if __name__ == "__main__":
    scrape_all_profiles()


🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=0
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=40
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=80
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=120
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=160
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=200
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=240
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=280
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=320
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=360
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=400
🔍 Fetching: https://www.polsinelli.com/people?search[post_type]=person&from=440
🔍 Fetching: https://www.polsinelli.com/peopl